# K-Ablation — Remaining 16 Configs

Runs exactly the configs missing or invalid from the previous run:
- **Qwen3-0.6B** — NQ k=1,5 | TriviaQA k=1,5 | HotpotQA k=1,5 (6 configs, fixes /nothink zero-output bug)
- **All other models** — HotpotQA k=1,5 only (10 configs)

**Fixed:** `chunk_size=256`, `retriever=faiss`, `k ∈ {1, 5}` only  
**No k=3** (already done in main experiment), **No k=10** (dropped)  

### Before running:
1. Runtime → **A100 GPU**
2. Corpus JSONs already on Drive from previous run — no re-upload needed
3. HuggingFace token in Colab Secrets as `HF_TOKEN`

In [ ]:
# ── Cell 1: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/chunk_slm_audit'
os.makedirs(f'{DRIVE_DIR}/results', exist_ok=True)
print('✅ Drive mounted')

In [ ]:
# ── Cell 2: Install Dependencies ──────────────────────────────────────────────
!pip install -q \
    "transformers>=4.44.0,<4.55.0" \
    "datasets>=2.20.0,<4.0.0" \
    "sentence-transformers>=2.7.0,<4.0.0" \
    "rank-bm25>=0.1.2" \
    "tiktoken>=0.7.0" \
    "wikipedia>=1.4.0" \
    "accelerate>=0.30.0" \
    "pandas>=2.0.0" \
    "tqdm"
print('✅ Dependencies installed')

In [ ]:
# ── Cell 3: HF Login & GPU Check ─────────────────────────────────────────────
from google.colab import userdata
import torch

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    print('✅ HF_TOKEN loaded')
except Exception:
    HF_TOKEN = ''
    print('⚠️  No HF_TOKEN — needed for Gemma-2-2B and Llama-3.2-1B')

assert torch.cuda.is_available(), '❌ GPU not available!'
print(f'✅ GPU: {torch.cuda.get_device_name(0)}  ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)')
DEVICE = 'cuda'

In [ ]:
# ── Cell 4: Config ────────────────────────────────────────────────────────────
import os
import torch

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

MODELS = {
    'SmolLM2-360M': {
        'model_id': 'HuggingFaceTB/SmolLM2-360M-Instruct',
        'dtype':    torch.float16,
        'trust_remote_code_model': False,
        'attn_implementation':     None,
    },
    'Qwen3-0.6B': {
        'model_id': 'Qwen/Qwen3-0.6B',
        'dtype':    torch.float16,
        'trust_remote_code_model': False,
        'attn_implementation':     None,
    },
    'Gemma-2-2B': {
        'model_id': 'google/gemma-2-2b-it',
        'dtype':    torch.bfloat16,
        'trust_remote_code_model': False,
        'attn_implementation':     None,
    },
    'Qwen3-1.7B': {
        'model_id': 'Qwen/Qwen3-1.7B',
        'dtype':    torch.float16,
        'trust_remote_code_model': False,
        'attn_implementation':     None,
    },
    'Llama-3.2-1B': {
        'model_id': 'meta-llama/Llama-3.2-1B-Instruct',
        'dtype':    torch.bfloat16,
        'trust_remote_code_model': False,
        'attn_implementation':     None,
    },
    'Phi-3.5-mini': {
        'model_id': 'microsoft/Phi-3.5-mini-instruct',
        'dtype':    torch.bfloat16,
        'trust_remote_code_model': False,
        'attn_implementation':     'eager',
    },
}

FIXED_CHUNK_SIZE = 256
FIXED_RETRIEVER  = 'faiss'
TOP_K_VALUES     = [1, 5]
NUM_SAMPLES      = 300

DATA_DIR    = f'{DRIVE_DIR}/data'
RESULTS_DIR = f'{DRIVE_DIR}/results'
PARTIAL     = f'{RESULTS_DIR}/k_ablation_remaining_partial.csv'
FINAL       = f'{RESULTS_DIR}/k_ablation_remaining.csv'

# ── Exactly which (model, dataset) pairs to run ───────────────────────────────
# Qwen3-0.6B: all 3 datasets (NQ+TriviaQA re-run due to zeros bug, HotpotQA new)
# All others: HotpotQA only
RUN_TARGETS = [
    ('Qwen3-0.6B',   'nq'),
    ('Qwen3-0.6B',   'triviaqa'),
    ('Qwen3-0.6B',   'hotpotqa'),
    ('SmolLM2-360M', 'hotpotqa'),
    ('Gemma-2-2B',   'hotpotqa'),
    ('Qwen3-1.7B',   'hotpotqa'),
    ('Llama-3.2-1B', 'hotpotqa'),
    ('Phi-3.5-mini', 'hotpotqa'),
]

# ── no-RAG baselines from main experiment (results.csv) ──────────────────────
# Used to compute distraction_ratio — no need to re-run baselines.
BASELINE_EM = {
    ('nq',       'SmolLM2-360M'): 0.0100,
    ('nq',       'Qwen3-0.6B'):   0.0267,
    ('nq',       'Gemma-2-2B'):   0.1067,
    ('nq',       'Qwen3-1.7B'):   0.0500,
    ('nq',       'Llama-3.2-1B'): 0.0400,
    ('nq',       'Phi-3.5-mini'): 0.0667,
    ('triviaqa', 'SmolLM2-360M'): 0.0233,
    ('triviaqa', 'Qwen3-0.6B'):   0.0833,
    ('triviaqa', 'Gemma-2-2B'):   0.2067,
    ('triviaqa', 'Qwen3-1.7B'):   0.1733,
    ('triviaqa', 'Llama-3.2-1B'): 0.1500,
    ('triviaqa', 'Phi-3.5-mini'): 0.3100,
    ('hotpotqa', 'SmolLM2-360M'): 0.0067,
    ('hotpotqa', 'Qwen3-0.6B'):   0.0533,
    ('hotpotqa', 'Gemma-2-2B'):   0.0533,
    ('hotpotqa', 'Qwen3-1.7B'):   0.0433,
    ('hotpotqa', 'Llama-3.2-1B'): 0.0167,
    ('hotpotqa', 'Phi-3.5-mini'): 0.0733,
}

print(f'Targets: {len(RUN_TARGETS)} (model, dataset) pairs × {len(TOP_K_VALUES)} k-values = {len(RUN_TARGETS)*len(TOP_K_VALUES)} configs')

In [ ]:
# ── Cell 5: Data Preparation ──────────────────────────────────────────────────
import json
import re
import time
from pathlib import Path
from tqdm.auto import tqdm
from datasets import load_dataset

def prepare_nq(n=NUM_SAMPLES):
    out = Path(DATA_DIR) / 'nq_corpus.json'
    if out.exists():
        print('[data_prep] Loading cached NQ corpus from Drive …')
        return json.loads(out.read_text())
    print('[data_prep] nq_corpus.json not found — fetching from HuggingFace + Wikipedia …')
    import wikipedia as _wp
    _wp.set_lang('en')
    def _fetch_wiki(question, answers):
        for q in [question] + list(answers[:2]):
            try:
                hits = _wp.search(q, results=4)
                for title in hits:
                    try:
                        page = _wp.page(title, auto_suggest=False)
                        if len(page.content.split()) >= 250:
                            return page.content
                    except (_wp.DisambiguationError, _wp.PageError):
                        continue
            except Exception:
                continue
        return None
    nq = load_dataset('google-research-datasets/nq_open', split='validation')
    samples, idx = [], 0
    with tqdm(total=n, desc='NQ') as pbar:
        while len(samples) < n and idx < len(nq):
            item = nq[idx]; idx += 1
            question, answers = item['question'], item['answer']
            if not answers: continue
            article = _fetch_wiki(question, answers)
            if article is None or len(article.split()) < 250: continue
            samples.append({'id': len(samples), 'dataset': 'nq',
                            'question': question, 'answers': answers, 'document': article})
            pbar.update(1); time.sleep(0.25)
    out.write_text(json.dumps(samples, indent=2, ensure_ascii=False))
    return samples

def prepare_triviaqa(n=NUM_SAMPLES):
    out = Path(DATA_DIR) / 'triviaqa_corpus.json'
    if out.exists():
        print('[data_prep] Loading cached TriviaQA corpus from Drive …')
        return json.loads(out.read_text())
    print('[data_prep] Loading TriviaQA …')
    ds = load_dataset('mandarjoshi/trivia_qa', 'rc', split='validation', trust_remote_code=True)
    samples = []
    with tqdm(total=n, desc='TriviaQA') as pbar:
        for item in ds:
            if len(samples) >= n: break
            answers = item['answer'].get('normalized_aliases') or [item['answer']['normalized_value']]
            answers = [a for a in answers if a.strip()]
            if not answers: continue
            wiki_contexts = [w for w in item.get('entity_pages', {}).get('wiki_context', []) if len(w.split()) >= 50]
            if not wiki_contexts: continue
            document = '\n\n'.join(wiki_contexts)
            if len(document.split()) < 250: continue
            samples.append({'id': len(samples), 'dataset': 'triviaqa',
                            'question': item['question'], 'answers': answers, 'document': document})
            pbar.update(1)
    out.write_text(json.dumps(samples, indent=2, ensure_ascii=False))
    return samples

def prepare_hotpotqa(n=NUM_SAMPLES):
    out = Path(DATA_DIR) / 'hotpotqa_corpus.json'
    if out.exists():
        print('[data_prep] Loading cached HotpotQA corpus from Drive …')
        return json.loads(out.read_text())
    print('[data_prep] Loading HotpotQA …')
    ds = load_dataset('hotpotqa/hotpot_qa', 'distractor', split='validation', trust_remote_code=True)
    samples = []
    with tqdm(total=n, desc='HotpotQA') as pbar:
        for item in ds:
            if len(samples) >= n: break
            answer = item['answer'].strip()
            if not answer: continue
            paragraphs = [t + '. ' + ' '.join(s)
                          for t, s in zip(item['context']['title'], item['context']['sentences'])]
            document = '\n\n'.join(paragraphs)
            if len(document.split()) < 100: continue
            samples.append({'id': len(samples), 'dataset': 'hotpotqa',
                            'question': item['question'], 'answers': [answer], 'document': document})
            pbar.update(1)
    out.write_text(json.dumps(samples, indent=2, ensure_ascii=False))
    return samples

def prepare_dataset(name, n=NUM_SAMPLES):
    return {'nq': prepare_nq, 'triviaqa': prepare_triviaqa, 'hotpotqa': prepare_hotpotqa}[name](n=n)

print('✅ Data prep loaded')

In [ ]:
# ── Cell 6: Chunker ───────────────────────────────────────────────────────────
import tiktoken
_ENC = tiktoken.get_encoding('cl100k_base')

def chunk_text(text, chunk_size, overlap=0):
    tokens = _ENC.encode(text)
    step   = chunk_size - overlap
    chunks = []
    for i in range(0, len(tokens), step):
        sl = tokens[i : i + chunk_size]
        if len(sl) < 10: break
        chunks.append(_ENC.decode(sl))
    return chunks

print('✅ Chunker loaded')

In [ ]:
# ── Cell 7: Dense Retriever ───────────────────────────────────────────────────
import numpy as np
from sentence_transformers import SentenceTransformer

_EMBED_MODEL = None
def _get_embed_model():
    global _EMBED_MODEL
    if _EMBED_MODEL is None:
        _EMBED_MODEL = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device='cpu')
    return _EMBED_MODEL

class DenseRetriever:
    def __init__(self, chunks):
        self.chunks = chunks
        model       = _get_embed_model()
        embs        = model.encode(chunks, batch_size=64, show_progress_bar=False, normalize_embeddings=True)
        self.embs   = embs.astype(np.float32)

    def retrieve(self, query, top_k=3):
        model   = _get_embed_model()
        q_emb   = model.encode([query], normalize_embeddings=True).astype(np.float32)
        scores  = self.embs @ q_emb.T
        top_idx = np.argsort(scores[:, 0])[::-1][:top_k]
        return [self.chunks[i] for i in top_idx]

# Retriever cache — built once per dataset, reused across all models
_retriever_cache  = {}
_dataset_samples  = {}

def get_retriever(dataset_name):
    if dataset_name not in _retriever_cache:
        samples = _dataset_samples[dataset_name]
        all_chunks = []
        for s in samples:
            all_chunks.extend(chunk_text(s['document'], FIXED_CHUNK_SIZE))
        print(f'[index] Building {dataset_name} retriever ({len(all_chunks)} chunks) …')
        _retriever_cache[dataset_name] = DenseRetriever(all_chunks)
        print(f'[index] Done')
    return _retriever_cache[dataset_name]

print('✅ Retriever loaded')

In [ ]:
# ── Cell 8: Metrics ───────────────────────────────────────────────────────────
import re, string

def _normalize(s):
    s = s.lower()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = s.translate(str.maketrans('', '', string.punctuation))
    return ' '.join(s.split())

def exact_match(pred, golds):
    return float(any(_normalize(pred) == _normalize(g) for g in golds))

def token_f1(pred, golds):
    p_toks = _normalize(pred).split()
    best = 0.0
    for g in golds:
        g_toks = _normalize(g).split()
        if not p_toks or not g_toks: continue
        common = set(p_toks) & set(g_toks)
        if not common: continue
        prec = len(common) / len(p_toks)
        rec  = len(common) / len(g_toks)
        best = max(best, 2 * prec * rec / (prec + rec))
    return best

def distraction_ratio(baseline_em, rag_em):
    if baseline_em <= 1e-9: return 0.0
    return max(0.0, (baseline_em - rag_em) / baseline_em)

print('✅ Metrics loaded')

In [ ]:
# ── Cell 9: Model Runner ──────────────────────────────────────────────────────
# FIX vs previous notebook: /nothink prefix REMOVED.
# Qwen3 thinking tokens are stripped by _THINK_RE instead.
# This matches the phi35 notebook approach that ran successfully.

import gc, shutil
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer

_phi3_cache = Path.home() / '.cache/huggingface/modules/transformers_modules/microsoft'
if _phi3_cache.exists():
    shutil.rmtree(_phi3_cache, ignore_errors=True)
    print('[setup] Cleared stale Phi3 module cache')

_model_cache, _tokenizer_cache = {}, {}

def _evict_all():
    for k in list(_model_cache):
        del _model_cache[k]; del _tokenizer_cache[k]
    gc.collect(); torch.cuda.empty_cache()

def load_model(model_name):
    if model_name in _model_cache:
        return _model_cache[model_name], _tokenizer_cache[model_name]
    _evict_all()
    cfg = MODELS[model_name]
    print(f'\n[runner] Loading {model_name} …')
    tok = AutoTokenizer.from_pretrained(cfg['model_id'], token=HF_TOKEN or None, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    kwargs = dict(torch_dtype=cfg['dtype'], token=HF_TOKEN or None,
                  trust_remote_code=cfg['trust_remote_code_model'])
    if cfg['attn_implementation']:
        kwargs['attn_implementation'] = cfg['attn_implementation']
    mdl = AutoModelForCausalLM.from_pretrained(cfg['model_id'], **kwargs).to(DEVICE)
    mdl.eval()
    _model_cache[model_name] = mdl; _tokenizer_cache[model_name] = tok
    return mdl, tok

_THINK_RE = re.compile(r'<think>.*?</think>', re.DOTALL)

def build_prompt(question, context, tokenizer):
    if context:
        user = (f'Context:\n{context}\n\n'
                f'Question: {question}\n'
                'Answer in a short phrase (1-5 words):')
    else:
        user = (f'Question: {question}\n'
                'Answer in a short phrase (1-5 words):')
    messages = [{'role': 'user', 'content': user}]
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        return user + '\n'

def generate_answer(model_name, question, context=None):
    mdl, tok = load_model(model_name)
    prompt   = build_prompt(question, context, tok)
    inputs   = tok(prompt, return_tensors='pt', truncation=True, max_length=2048).to(DEVICE)
    with torch.no_grad():
        out = mdl.generate(**inputs, max_new_tokens=50,
                           do_sample=False, pad_token_id=tok.eos_token_id)
    new_ids = out[0][inputs['input_ids'].shape[1]:]
    raw = tok.decode(new_ids, skip_special_tokens=True).strip()
    raw = _THINK_RE.sub('', raw).strip()
    for line in raw.splitlines():
        if line.strip(): return line.strip()
    return raw

print('✅ Model runner loaded')

In [ ]:
# ── Cell 10: Resume & Experiment Loop ─────────────────────────────────────────
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

def _flush():
    gc.collect(); torch.cuda.synchronize(); torch.cuda.empty_cache()

def _load_done():
    frames = []
    for p in (FINAL, PARTIAL):
        if Path(p).exists(): frames.append(pd.read_csv(p))
    if not frames: return set(), []
    df   = pd.concat(frames, ignore_index=True).drop_duplicates(
        subset=['dataset', 'model', 'chunk_size', 'retriever', 'top_k'])
    keys = {(r['dataset'], r['model'], str(r['chunk_size']), r['retriever'], str(int(r['top_k'])))
            for _, r in df.iterrows()}
    return keys, df.to_dict('records')

def _save(results):
    pd.DataFrame(results).to_csv(PARTIAL, index=False)


def run_experiment():
    done, results = _load_done()
    print(f'Resuming — {len(done)} configs already done')

    # Pre-load all needed dataset samples
    needed_datasets = list(dict.fromkeys(ds for _, ds in RUN_TARGETS))
    for ds in needed_datasets:
        print(f'[data] Loading {ds} …')
        _dataset_samples[ds] = prepare_dataset(ds)[:NUM_SAMPLES]

    # Loop by model (load each model once, run all its datasets)
    model_to_datasets = {}
    for model_name, dataset_name in RUN_TARGETS:
        model_to_datasets.setdefault(model_name, []).append(dataset_name)

    for model_name, datasets in model_to_datasets.items():
        print(f'\n{"="*60}')
        print(f'MODEL: {model_name}  |  datasets: {datasets}')
        print(f'{"="*60}')

        for dataset_name in datasets:
            samples   = _dataset_samples[dataset_name]
            retriever = get_retriever(dataset_name)
            base_em   = BASELINE_EM[(dataset_name, model_name)]

            for top_k in TOP_K_VALUES:
                key = (dataset_name, model_name, str(FIXED_CHUNK_SIZE), FIXED_RETRIEVER, str(top_k))
                if key in done:
                    print(f'  [skip] {dataset_name} k={top_k}')
                    continue

                em_list, f1_list = [], []
                desc = f'  [{dataset_name} k={top_k}] {model_name}'
                for idx, s in enumerate(tqdm(samples, desc=desc, leave=False)):
                    try:
                        context = '\n\n'.join(retriever.retrieve(s['question'], top_k=top_k))
                        pred    = generate_answer(model_name, s['question'], context=context)
                    except Exception as exc:
                        print(f'\n  [WARN] sample {s["id"]} failed: {exc}')
                        pred = ''
                    em_list.append(exact_match(pred, s['answers']))
                    f1_list.append(token_f1(pred, s['answers']))
                    if (idx + 1) % 10 == 0: _flush()

                avg_em = sum(em_list) / len(em_list)
                avg_f1 = sum(f1_list) / len(f1_list)
                row = {
                    'dataset': dataset_name, 'model': model_name,
                    'chunk_size': FIXED_CHUNK_SIZE, 'retriever': FIXED_RETRIEVER, 'top_k': top_k,
                    'em':  round(avg_em, 4), 'f1': round(avg_f1, 4),
                    'distraction_ratio': round(distraction_ratio(base_em, avg_em), 4),
                    'n_samples': len(samples),
                }
                results.append(row); done.add(key)
                _save(results); _flush()
                print(f'  {dataset_name} k={top_k}  EM={avg_em:.3f}  F1={avg_f1:.3f}  '
                      f'DR={row["distraction_ratio"]:.3f}  [saved]')

    df = pd.DataFrame(results)
    df.to_csv(FINAL, index=False)
    if Path(PARTIAL).exists(): Path(PARTIAL).unlink()
    print(f'\n✅ DONE — {len(df)} rows saved to {FINAL}')
    return df

print('✅ Experiment loop ready')

In [ ]:
# ── Cell 11: RUN ──────────────────────────────────────────────────────────────
missing = [d for d in ['nq', 'triviaqa', 'hotpotqa']
           if not Path(f'{DATA_DIR}/{d}_corpus.json').exists()]
if missing:
    print(f'⚠️  Missing corpus files: {missing} — upload to {DATA_DIR}/')
else:
    print('✅ All corpus files found. Starting …\n')
    df = run_experiment()
    print(df.to_string())

In [ ]:
# ── Cell 12: Download Results ─────────────────────────────────────────────────
from google.colab import files
files.download(FINAL)